- https://huggingface.co/blog/NormalUhr/rlhf-pipeline
    - https://github.com/brevdev/launchables/blob/main/llama3dpo.ipynb
- https://gemini.google.com/app/f77054cddf99235c

> gradient 是 learning 最核心本质的 driven force，learning signal

In [7]:
from IPython.display import Image

In [8]:
Image(url='./imgs/rl-loss-dpo.jpeg', width=800)

- rl4llm（rlhf，rlvr，rft）：奖励即回报，reward model 是定义在 whole response 上的
- dpo 不再单独训练 reward model，直接用 BT model 的建模思路，直接训练 $\pi_\theta$ （生成模型，$\pi$ 和 reward model 之间存在关系）
    - 本来 bt model 是用来训练 reward model，dpo 建模思路下消掉了 reward model

$$
\max_{\pi} \mathbb{E}_{x \sim \mathcal{D}} \left[ \mathbb{E}_{y \sim \pi(\cdot|x)} [r(x, y)] - \beta D_{KL}(\pi(\cdot|x) || \pi_{ref}(\cdot|x)) \right]
$$

- KL-constrained reward maximization objective
- The added constraint is important, as it prevents the model from deviating too far from **the distribution on which the reward model is accurate**, as well as maintaining the generation diversity and preventing mode-collapse to single high-reward answers. (DPO)
    - "它能防止模型过分偏离奖励模型能够准确评估的数据分布，同时还能保持生成内容的多样性，并避免因‘模式坍塌’而收敛到单一的高奖励答案。"
    - 奖励模型（trained/learned Reward Model）是基于特定的数据分布训练出来的。如果生成模型跑偏了，产生了一些它没见过的数据（Out-of-distribution），奖励模型打出的分数就不准了（可能会乱给高分）。
- kl reg: Rafailov 是 Sergey Levine 的学生，Levine 也是 MaxEnt RL 的推手

### DPO insights

- 即找一个策略 $\pi$，让它生成的回答 $y$ 分数 $r$ 最高，但同时不能偏离参考模型 $\pi_{ref}$ 太远。

$$
\pi^*(y|x) = \frac{1}{Z(x)} \pi_{ref}(y|x) \cdot e^{\frac{r(x,y)}{\beta}}
$$

- 即上述目标函数存在一个解析解（Closed-form solution/Analytical Solution）
    - https://github.com/chunhuizhang/personal_chatgpt/blob/main/tutorials/trl_hf/dpo_math.ipynb
    - 最优策略 = 参考策略 $\times$ 奖励的指数加权。
    - 把参考策略 $\pi_{ref}$ 按照奖励 $r$ 进行缩放（奖励高的地方概率放大，奖励低的地方概率缩小）。
    - $Z(x)$ 是一个归一化常数（Partition Function），确保概率加起来等于 1。
- 最优策略 $\pi^*$ 和奖励 $r$ 是一一对应的，那我能不能把 $r$ 反解出来？
- $r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$
    - Reward 被表示成了 $\pi$ 和 $\pi_{ref}$ 的 Log Ratio（即 KL 的核心项）。
    - 在 KL 散度约束下的强化学习最优策略 $\pi^*$ 和奖励函数 $r^*$ 之间存在解析映射关系
- 代入 Bradley-Terry 偏好模型：$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$
$$
\begin{aligned}
r(x, y_w) - r(x, y_l) &= \left( \beta \log \frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} + \beta \log Z(x) \right) - \left( \beta \log \frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)} + \beta \log Z(x) \right) \\
&= \beta \log \frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)}
\end{aligned}
$$
- 数学的 magic 时刻
    - 那个麻烦的归一化常数 $Z(x)$ 因为做差被抵消了！
    - 显式的奖励函数 $r(x,y)$ 消失了！
- 一些 insights
    - DPO 把强化学习问题，强行降维成了一个二分类的监督学习问题（BT modeling）。它其实就是我们最熟悉的逻辑回归（Logistic Regression）或者是二元交叉熵（Binary Cross Entropy）。
    - $u=\text{分差} = \beta \log \frac{\pi(y_w)}{\pi_{ref}(y_w)} - \beta \log \frac{\pi(y_l)}{\pi_{ref}(y_l)}$
    - $P(\text{win}) = \sigma(u)$
    - 真实标签：在数据集里，$y_w$ 确实赢了（标签为 1）。模型预测：模型认为 $y_w$ 赢的概率是 $\sigma(u)$。目标：我们要调整参数 $\theta$，让这个预测概率 $\sigma(u)$ 最大化（越接近 1 越好）。$$\text{Goal: } \max_\theta \sigma(u)$$

$$
\mathcal{L}_{\text{DPO}}(\pi_{\theta}; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi_{\theta}(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_{\theta}(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right].
$$

### Discussion

- Does the model have the capacity to improve both its **"evaluation"** ($R(x,y)$) and **"generation"**($\pi_\theta$) abilities simultaneously? In other words, DPO's training process focuses solely on teaching the model **how to "score" responses**, akin to learning how to assess moves from game records. However, this does not necessarily ensure that the model will make optimal decisions during actual generation—**it does not prove that evaluation ability will seamlessly translate into effective generative performance**. If this assumption fails, then the DPO training process loses its purpose. Just as one does not expect to become a chess master solely by reading game records, the validity of this assumption also directly impacts the rationale behind other methods like SPIN and self-reward.
    - 模型是否具备同时提升其“评估”与“生成”能力的潜力？换言之，DPO 的训练过程仅专注于教会模型如何为回复“打分”，这类似于通过研读棋谱来学习如何评估招法。然而，这并不一定能保证模型在实际生成阶段做出最优决策——即无法证明“评估”能力可以无缝转化为有效的“生成”表现。倘若这一假设失效，DPO 的训练过程便失去了意义。正如人们不指望仅凭研读棋谱就能成为国际象棋大师，这一假设的有效性也直接关系到 SPIN 和 Self-Reward 等其他方法的理论根基。
- Moreover, because the DPO optimization objective depends exclusively on the reward model's scoring, it only cares about whether the change in scores relative to the reference model meets expectations, not whether the generated sentences are fluent or appealing. In other words, DPO focuses more on enlarging the **loss margin** rather than on ensuring that the model **generates high-quality outputs**. This can lead to an awkward phenomenon during DPO training: the losses for both good and bad responses may increase simultaneously, forcing us to adjust hyperparameters or add additional constraints to stabilize training.
    - 由于 DPO 的优化目标完全依赖于奖励模型的评分机制，它只关注相对于参考模型的分数变化是否符合预期，而不在乎生成的句子是否通顺或具有吸引力。换句话说，DPO 更多地是专注于扩大“损失边际”（loss margin），而不是确保模型生成高质量的输出。这会导致 DPO 训练过程中出现一个尴尬的现象：好回复和坏回复的损失可能会同时增加，迫使我们不得不调整超参数或添加额外的约束来稳定训练。
    - 好回复增加的更多，坏回复也增加，只是增加得更少
- 为什么 DPO 容易 Reward Hacking（过拟合）？因为在 DPO 的 Loss 里，只要 $\frac{\pi(y_w)}{\pi_{ref}(y_w)}$ 足够大，Loss 就会很低。模型可以通过把参考模型 $\pi_{ref}$ 认为概率很低的地方强行提权，从而制造巨大的 Log Ratio。这本质上是在利用 KL 散度的定义漏洞。